# Model: Basic CNN\n**Dataset:** CIFAR-10\n**Assignment Context:** Training CNNs from scratch on CIFAR-10.

**Học viện Công nghệ Bưu chính Viễn thông (PTIT) — Khoa CNTT 1**
- **Sinh viên:** Nguyễn Nam Hải (B23DCCN277 - D23CTPM01 - CT01)
- **GVHD:** PGS.TS. Trần Đình Quế
- **GitHub Repository:** [https://github.com/HandQ2212/intel-sys-assignment-05](https://github.com/HandQ2212/intel-sys-assignment-05)
- **Kaggle Dataset:** [CIFAR-10 Image Classification Dataset](https://www.kaggle.com/datasets/ayush1220/cifar10)


## 2. Cơ sở lý thuyết

**Mạng Convolutional Neural Network (CNN) cơ bản**

- **Phép toán Tích chập (Convolution):**
  Công thức tích chập 2D:
  $$Y(i,j) = \sum_{u} \sum_{v} K(u,v) \cdot X(i+u,j+v) + b$$
  Trong đó $X$ là ảnh đầu vào hoặc feature map, $K$ là kernel, $b$ là bias.
- **Tính chất:**
  - *Kết nối cục bộ (Local connectivity):* Mỗi neuron chỉ kết nối với một vùng nhỏ của đầu vào.
  - *Chia sẻ trọng số (Weight sharing):* Kernel được sử dụng chung trên toàn bộ ảnh, giúp giảm số lượng tham số.
- **Các thành phần khác:**
  - *Hàm kích hoạt ReLU:* $$f(x) = \max(0, x)$$
  - *Batch Normalization (BN):* Chuẩn hóa đầu ra của một lớp để quá trình huấn luyện ổn định hơn.
  - *Pooling:* Giảm kích thước không gian (spatial size) của feature map.
- **Hợp thành hàm (Function composition):**
  Một mạng CNN có thể coi như hợp thành của nhiều hàm:
  $$\hat{y} = f_{FC} \circ f_{pool2} \circ f_{relu2} \circ f_{conv2} \circ f_{pool1} \circ f_{relu1} \circ f_{conv1}(X)$$


## 3. Imports and Setup

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')


## 4. Configuration

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64
LR = 1e-3
IN_CHANNELS = 3
NUM_CLASSES = 10
MODEL_NAME = 'Basic CNN'
DATASET_NAME = 'CIFAR-10'
MODEL_KEY = 'basic'


## 5. Data Loading

In [ ]:
# Data transforms
train_transform = transforms.Compose([
    transforms.Resize(32),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

test_transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

# Load dataset
train_dir = '../intel-sys-assignment-04/dataset/cifar10/train/'
test_dir = '../intel-sys-assignment-04/dataset/cifar10/test/'

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print(f'Classes: {class_names}')
print(f'Train size: {len(train_dataset)}, Test size: {len(test_dataset)}')


## 6. Data Exploration

In [ ]:
# Plot class distribution
train_counts = [0] * NUM_CLASSES
for _, label in train_dataset:
    train_counts[label] += 1

plt.figure(figsize=(10, 5))
plt.bar(class_names, train_counts)
plt.title('Class Distribution in Training Set')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

# Show a grid of sample images (one per class)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()
found_classes = set()

# To get unnormalized images for visualization
inv_normalize = transforms.Normalize(
    mean=[-0.4914/0.2470, -0.4822/0.2435, -0.4465/0.2616],
    std=[1/0.2470, 1/0.2435, 1/0.2616]
)

for img, label in train_dataset:
    if label not in found_classes:
        found_classes.add(label)
        img = inv_normalize(img)
        img = img.numpy().transpose(1, 2, 0)
        img = np.clip(img, 0, 1)
        axes[label].imshow(img)
        axes[label].set_title(class_names[label])
        axes[label].axis('off')
    if len(found_classes) == NUM_CLASSES:
        break
plt.tight_layout()
plt.show()


## 7. Model Architecture

In [ ]:
class BasicCNN(nn.Module):
    """
    M1: Basic CNN
    ŷ = FC(Pool(ReLU(BN(Conv₂(Pool(ReLU(BN(Conv₁(X)))))))))
    """
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model_class = BasicCNN


## 8. Training Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


## 9. Training Loop

In [ ]:
# Instantiate model, loss, optimizer
model = model_class(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

train_losses, train_accs = [], []
test_losses, test_accs = [], []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    print(f'Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


## 10. Visualization

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(test_accs, label='Test Acc')
plt.title('Accuracy over Epochs')
plt.legend()
plt.show()


## 11. Final Evaluation

In [ ]:
test_loss, test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, device)

print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


## 12. Save Results

In [ ]:
os.makedirs('results', exist_ok=True)
results = {
    'model': MODEL_NAME,
    'dataset': DATASET_NAME,
    'test_loss': test_loss,
    'test_acc': test_acc,
    'train_losses': train_losses,
    'train_accs': train_accs,
    'test_losses': test_losses,
    'test_accs': test_accs
}
with open(f'results/cifar10_{MODEL_KEY}.json', 'w') as f:
    json.dump(results, f, indent=4)
print(f"Results saved to results/cifar10_{MODEL_KEY}.json")


## 13. Conclusion\nModel trained successfully.